# 4.1 Your First LLM Call: Concepts Notebook

**Generative AI Foundations · Module 4**

This notebook explains each idea in small, simple steps. Run the cells from top to bottom.

| # | Concept |
|---|---|
| 1 | Settings and secrets (environment variables) |
| 2 | The `provider:model` name |
| 3 | Creating a model with `init_chat_model` |
| 4 | Your first call: text in, message out |
| 5 | What is inside the reply |
| 6 | Messages and roles |
| 7 | Memory: how chat history works |
| 8 | Streaming |
| 9 | Switching providers |

**Before you start:** Ollama is running, and you have pulled the model:

```bash
ollama pull gemma3:1b
```

## 0. Install the Packages

Run this once. (In a `uv` project you can use `uv add "langchain[ollama,google-genai]" python-dotenv` instead.)

In [ ]:
%pip install -q "langchain[ollama,google-genai]" python-dotenv

---
## 1. Settings and Secrets

> **Mental Model: Labelled Jars**
> Environment variables are labelled jars on a shelf outside your code. Your code says "give me what's in the jar labelled `MODEL_NAME`". To change the setting, you change the jar, not the code.

Why? Because API keys are **secrets**. If you type a key into your code and share the code, you share the key.

In [ ]:
import os

# Put a value in a jar
os.environ["MODEL_NAME"] = "ollama:gemma3:1b"

# Read it back
print(os.getenv("MODEL_NAME"))

Reading a jar that does not exist gives `None`, so we usually give a **default**:

In [ ]:
print(os.getenv("SOME_SETTING"))           # None: the jar does not exist
print(os.getenv("TEMPERATURE", "0.7"))     # "0.7": the default is used

**Important:** values from environment variables are always **text (strings)**. Convert numbers yourself:

In [ ]:
temperature = os.getenv("TEMPERATURE", "0.7")
print(type(temperature), temperature)

temperature = float(temperature)
print(type(temperature), temperature)

In real projects, settings live in a `.env` file, and `load_dotenv()` fills the jars from it. If there is no `.env` file, nothing happens, so this cell is always safe to run:

In [ ]:
from dotenv import load_dotenv

load_dotenv()
print("MODEL_NAME =", os.getenv("MODEL_NAME"))

---
## 2. The `provider:model` Name

> **Mental Model: Company and Product**
> `ollama:gemma3:1b` is like `Toyota:Corolla`. The part **before the first colon** is the company (provider). Everything **after** it is the product (model).

```text
ollama : gemma3:1b
  |          |
provider   model
```

In [ ]:
names = [
    "ollama:gemma3:1b",
    "google_genai:gemini-3.5-flash-lite",
    "anthropic:claude-sonnet-5-5",
]

for name in names:
    provider, model = name.split(":", 1)   # split only at the FIRST colon
    print(f"{provider:<14} -> {model}")

Notice `gemma3:1b` keeps its own colon, because we split only once.

---
## 3. Creating a Model with `init_chat_model`

> **Mental Model: The Universal Power Adapter**
> Every provider has a different "socket". `init_chat_model` is one adapter that fits them all. You give it the `provider:model` name, and it builds the right model object for you.

Creating a model does **not** call the AI yet. It only prepares the connection.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("ollama:gemma3:1b", temperature=0.7)

print(type(llm).__name__)
print(llm.model)
print(llm.temperature)

The same function with a different name builds a **different class**. (Gemini needs an API key, so this example only runs if you have set one.)

In [ ]:
if os.getenv("GOOGLE_API_KEY"):
    gemini = init_chat_model("google_genai:gemini-3.5-flash-lite")
    print(type(gemini).__name__)
else:
    print("No GOOGLE_API_KEY set: skipping the Gemini example.")

---
## 4. Your First Call: Text In, Message Out

> **Mental Model: Sending a Letter**
> You post a letter (`invoke`) and get a letter back. The reply is not just text; it is a full **message object** with the text inside, plus details on the envelope.

```text
"What is Python?"  --->  [ model ]  --->  AIMessage(text="Python is ...")
```

In [ ]:
response = llm.invoke("In one sentence, what is Python?")

print(response.text)

**Example 2:** a different question, same pattern.

In [ ]:
response = llm.invoke("Give me one tip for learning to code. One sentence.")
print(response.text)

---
## 5. What Is Inside the Reply?

The reply is an `AIMessage`. Think of it as a letter in an envelope:

| Part | What it holds |
|---|---|
| `.text` | The answer text (the letter) |
| `.usage_metadata` | Tokens used (the postage you pay) |
| `.response_metadata` | Model name, provider, finish reason (the stamps on the envelope) |

In [ ]:
response = llm.invoke("Say hello in three words.")

print("Type:    ", type(response).__name__)
print("Text:    ", response.text)
print("Tokens:  ", response.usage_metadata)
print("Model:   ", response.response_metadata.get("model_name"))
print("Provider:", response.response_metadata.get("model_provider"))

---
## 6. Messages and Roles

A conversation is a **list of messages**. Each message has a **role**:

| Role | Who speaks | Example |
|---|---|---|
| `system` | You, the developer: the rules | "You are a friendly tutor." |
| `user` | The person using the app | "What is Python?" |
| `assistant` | The AI's earlier replies | "Python is a language..." |

> **Mental Model: A Play Script**
> The **system** message is the director's note. The **user** and **assistant** messages are the actors' lines.

In [ ]:
messages = [
    {"role": "system", "content": "You are a friendly teacher. Answer in one sentence."},
    {"role": "user", "content": "What is Python?"},
]

print(llm.invoke(messages).text)

**Example 2:** same question, different system message, different style.

In [ ]:
messages = [
    {"role": "system", "content": "You are a pirate. Answer in one sentence."},
    {"role": "user", "content": "What is Python?"},
]

print(llm.invoke(messages).text)

---
## 7. Memory: How Chat History Works

**The model remembers nothing between calls.** Each call starts fresh.

> **Mental Model: The Goldfish With a Notebook**
> The model is a goldfish: it forgets everything after each answer. The **history list** is a notebook you hand it before every question.

### 7a. Without History: It Forgets

In [ ]:
llm.invoke("Hi, my name is Ali.")

reply = llm.invoke("What is my name?")
print(reply.text)

Two separate calls. The second call never saw "my name is Ali".

### 7b. With History: It Remembers

```text
Turn 1   history = [system, user: "I'm Ali"]                      -> model -> "Hello Ali!"
         history = [system, user: "I'm Ali", assistant: "Hello Ali!"]

Turn 2   history = [system, user: "I'm Ali", assistant: "Hello Ali!",
                    user: "What is my name?"]                     -> model -> "Your name is Ali."
```

Every turn: **add the question, call the model with the whole list, add the answer.**

In [ ]:
history = [{"role": "system", "content": "You are a helpful assistant. Keep answers short."}]

# Turn 1
history.append({"role": "user", "content": "Hi, my name is Ali."})
reply = llm.invoke(history)
history.append(reply)              # save the AI's answer

# Turn 2
history.append({"role": "user", "content": "What is my name?"})
reply = llm.invoke(history)
history.append(reply)

print(reply.text)

Look at the notebook (the history list). It grows by two messages every turn:

In [ ]:
from langchain_core.messages.utils import convert_to_messages

for message in convert_to_messages(history):
    print(f"{message.type:<7} {message.text}")

### 7c. A Common Mistake

Saving the reply as **plain text** instead of the message:

```python
history.append(reply.text)   # wrong
history.append(reply)        # right
```

Plain text has no role, so LangChain treats it as a **user** message. The model then thinks *you* said its answer.

In [ ]:
wrong = [
    {"role": "user", "content": "Hi, my name is Ali."},
    "Hello Ali! How can I help?",          # reply saved as plain text (the mistake)
]

for message in convert_to_messages(wrong):
    print(f"{message.type:<7} {message.text}")

Both lines show `human`. The AI's answer was stored as if the user said it.

---
## 8. Streaming

Normally you wait for the full answer. **Streaming** shows the answer piece by piece as it is written.

> **Mental Model: A Tap, Not a Bucket**
> `invoke` gives you a full bucket of water at the end. `stream` is a tap: water flows as soon as it is ready.

```text
invoke:  ..........wait..........  "Python is a popular language."
stream:  "Python" " is" " a" " popular" " language."   (appears word by word)
```

In [ ]:
for chunk in llm.stream("In one sentence, what is Python?"):
    print(chunk.text, end="|", flush=True)

The `|` marks show each separate chunk. Remove `"|"` (use `end=""`) to see normal text.

To keep the **full** message (for example, to save it in history), add the chunks together:

In [ ]:
full = None
for chunk in llm.stream("In one sentence, what is Python?"):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk

print()
print("\nFull text:", full.text)

---
## 9. Switching Providers

Your code stays the same. Only the **name** changes.

> **Mental Model: Change the Socket, Keep the Laptop**

In [ ]:
def ask(model_name: str, question: str) -> str:
    model = init_chat_model(model_name, temperature=0.7)
    return model.invoke(question).text


print(ask("ollama:gemma3:1b", "In five words, what is AI?"))

if os.getenv("GOOGLE_API_KEY"):
    print(ask("google_genai:gemini-3.5-flash-lite", "In five words, what is AI?"))

---
## Summary

| Concept | One line to remember |
|---|---|
| Environment variables | Settings and secrets live outside your code; values are text |
| `provider:model` | Company before the first colon, model after it |
| `init_chat_model` | One adapter that builds the right model for any provider |
| `invoke` | Text or messages in, an `AIMessage` out |
| `AIMessage` | `.text`, `.usage_metadata`, `.response_metadata` |
| Roles | `system` sets rules; `user` asks; `assistant` answered before |
| History | The model forgets; you resend the list and append the reply message |
| Streaming | Chunks arrive one by one; add them together for the full message |

## Practice

1. Change the system message in Section 6 to "You are a cricket commentator." What changes?
2. In Section 7b, add a third turn: "Where do I live?" (after telling it your city). Does it remember?
3. In Section 8, count how many chunks a reply has.